In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_percentage_error

demand_df = pd.read_excel("./Dataset/PGCB_date_power_demand.xlsx")
weather_df = pd.read_excel("./Dataset/weather_data.xlsx", skiprows=3)

In [ ]:
# Cutoff date between train and test data
cutoff_date = '2024-01-01'

In [ ]:
# Setting up the index
demand_df['DT'] = pd.to_datetime(demand_df['datetime'])
demand_df = demand_df.set_index('DT')
demand_df = demand_df.sort_index()

# Copying the original dataframe
original_demand_df = demand_df.copy()

# Cleaning the anomalies (reasonable values are from 3000 to 50000)
demand_df.loc[(demand_df.index < cutoff_date) & (demand_df['demand_mw'] < 3000), 'demand_mw'] = np.nan
demand_df.loc[(demand_df.index < cutoff_date) & (demand_df['demand_mw'] > 50000), 'demand_mw'] = np.nan

# Base clean data column
demand_df['clean_demand_mw'] = demand_df['demand_mw']

# Interpolating missing values based on time, only training slice
demand_df.loc[demand_df.index < cutoff_date, 'clean_demand_mw'] = demand_df.loc[demand_df.index < cutoff_date, 'clean_demand_mw'].interpolate(method='time')

# # Displays the cleaned data
# # fig = px.line(demand_df, x='datetime', y='demand_mw', title='Raw Grid Power Demand')
# # fig.show()

# # display(demand_df.head())

In [ ]:
final_df = demand_df[['clean_demand_mw']].copy()

# Building potentially useful features
final_df['hour'] = np.sin(2 * np.pi * final_df.index.hour / 24)
final_df['day_of_week'] = np.sin(2 * np.pi * final_df.index.dayofweek / 7)
final_df['month'] = final_df.index.month
final_df['is_weekend'] = ((final_df.index.dayofweek == 4) | (final_df.index.dayofweek == 5)).astype(int) # In Bangladesh weekends are on Fri / Sat

# Demand x hours ago is saved as demand_lag_x
final_df['demand_lag_1'] = final_df['clean_demand_mw'].shift(1) # 1 hour
final_df['demand_lag_2'] = final_df['clean_demand_mw'].shift(2) # 2 hours
final_df['demand_lag_24'] = final_df['clean_demand_mw'].shift(24) # 1 day
final_df['demand_lag_168'] = final_df['clean_demand_mw'].shift(168) # 1 week

# Average demand of previous 6 hours (excluding the current hour)
final_df['rolling_mean_3h'] = final_df['clean_demand_mw'].shift(1).rolling(window=3).mean()
final_df['rolling_mean_6h'] = final_df['clean_demand_mw'].shift(1).rolling(window=6).mean()
final_df['rolling_mean_9h'] = final_df['clean_demand_mw'].shift(1).rolling(window=9).mean()
final_df['rolling_mean_12h'] = final_df['clean_demand_mw'].shift(1).rolling(window=12).mean()
final_df['rolling_mean_24h'] = final_df['clean_demand_mw'].shift(1).rolling(window=24).mean()
final_df['rolling_mean_168h'] = final_df['clean_demand_mw'].shift(1).rolling(window=168).mean()

# Standard Deviation over previous 12 hours (excluding the current hour)
final_df['rolling_std_6h'] = final_df['clean_demand_mw'].shift(1).rolling(window=6).std()
final_df['rolling_std_12h'] = final_df['clean_demand_mw'].shift(1).rolling(window=12).std()
final_df['rolling_std_24h'] = final_df['clean_demand_mw'].shift(1).rolling(window=24).std()
final_df['rolling_std_168h'] = final_df['clean_demand_mw'].shift(1).rolling(window=168).std()

# Building Target
final_df['target_t_plus_1'] = final_df['clean_demand_mw'].shift(-1)

# Drop all rows that have NaNs due to lagging and rolling calculations
final_df = final_df.dropna()

# # Display cleaned dataframe
# # display(final_df.head(n=30))

In [ ]:
# Setting up the index
weather_df['DT'] = pd.to_datetime(weather_df['time'])
weather_df = weather_df.set_index('DT')
weather_df = weather_df.sort_index()

# Selecting important columns and joining
weather_features = weather_df[['temperature_2m (°C)', 'apparent_temperature (°C)', 'relative_humidity_2m (%)', 'precipitation (mm)']]
final_df = final_df.join(weather_features, how='left')

# Handle missing weather data, dropping if it doesn't work
final_df[weather_features.columns] = final_df[weather_features.columns].ffill()
final_df = final_df.dropna()

# # The Final Dataframe
# # display(final_df.head())

In [ ]:
# Separating train and test data
train_df = final_df.loc[final_df.index < cutoff_date]
test_df = final_df.loc[final_df.index >= cutoff_date]

# Separating the Features (X) from the Target (y)
X_train = train_df.drop(columns=['target_t_plus_1'])
y_train = train_df['target_t_plus_1']
X_test = test_df.drop(columns=['target_t_plus_1'])
y_test = test_df['target_t_plus_1']

# If I feel that the data provided to me is incorrect can I estimate the correct data and use it to model the prediction? Didn't implement as I wasn't sure...
# X_test_alt = X_test.copy()
# X_test_alt.loc[(X_test_alt['clean_demand_mw'] > 50000) | (X_test_alt['clean_demand_mw'] < 3000), 'clean_demand_mw'] = np.nan
# X_test_alt['clean_demand_mw'] = X_test_alt['clean_demand_mw'].fillna(X_test_alt['rolling_mean_3h'])

GridSearch Model: gives best parameters as written in the next cell (run if required)

In [ ]:
# from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
# import joblib

# tscv = TimeSeriesSplit(n_splits=3)

# param_grid = {
#     'n_estimators': [400],           
#     'max_depth': [None, 10],          
#     'min_samples_leaf': [1, 2],        
#     'max_features': [0.7, 0.8, 0.9, 1.0]  
# }

# rf_base = RandomForestRegressor(random_state=42) 

# grid_search = GridSearchCV(
#     estimator=rf_base,
#     param_grid=param_grid,
#     cv=tscv,
#     scoring='neg_mean_absolute_percentage_error',
#     n_jobs=4,
#     verbose=3
# )

# grid_search.estimator.n_jobs = 4
# grid_search.n_jobs = 4

# with joblib.parallel_backend('threading'):
#     grid_search.fit(X_train, y_train)

# best_rf = grid_search.best_estimator_
# print(f"\nBest Parameters Found: {grid_search.best_params_}")

In [ ]:
# The core Random Forest model, with hyperparameters optimized using Grid Search
rf_model = RandomForestRegressor(n_estimators=400, random_state=69, n_jobs=-1, max_features=0.8, max_depth=None, min_samples_leaf=2)

# Training model
rf_model.fit(X_train, y_train)

# Testing model
y_pred = rf_model.predict(X_test)

mape = mean_absolute_percentage_error(y_test, y_pred)

# Final MAPE
print(f"Baseline Random Forest Test MAPE: {mape * 100:.2f}%")

In [ ]:
# Extracting feature importances
importances = rf_model.feature_importances_

feature_importance_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': importances
})

feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)

# Visualize and print the feature importances
fig = px.bar(
    feature_importance_df, 
    x='Importance', 
    y='Feature', 
    orientation='h',
    title='Random Forest Feature Importances - Key Drivers of Demand',
    color='Importance',
    color_continuous_scale='Viridis'
)

fig.update_layout(showlegend=False, xaxis_title="Relative Importance (Gini)", yaxis_title="")
fig.show()

display(feature_importance_df.head(n=100))